In [34]:
import os, re, pickle
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")
RANDOM_STATE = 42

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [35]:
jd_df_raw = pd.read_excel(r"C:\Users\anish\Desktop\Mp - Refs\MajorProject_Code\datasets\jobs_dataset_full.xlsx")
# resumes will be built from txt files instead of a CSV

In [36]:
jd_df_raw.head()

,job_title,company,location,description
0,Cybersecurity Analyst,FIS Global,"Bengaluru, Karnataka, India",FIS Global is hiring a Cybersecurity Analyst f...
1,Associate Cybersecurity Analyst,Unisys,"Bengaluru, Karnataka, India",Unisys is hiring Associate Cybersecurity Analy...
2,Junior Cyber Security Analyst,Endava,"Bengaluru, Karnataka, India",Endava is looking for a Junior Cyber Security ...
3,Junior Security Analyst,GrowthsphereAI Private Limited,"Bengaluru, Karnataka, India",GrowthsphereAI Private Limited in Bengaluru is...
4,Cybersecurity Analyst,Barracuda Networks,"Bengaluru, Karnataka, India",Barracuda Networks is seeking a Cybersecurity ...


In [37]:
import os
import glob
import pandas as pd

# Updated path using a raw string (r"...") to handle Windows backslashes safely
RESUME_DIR = r"C:\Users\anish\Desktop\Mp - Refs\MajorProject_Code\datasets\convertedresumes"

resume_records = []

for path in glob.glob(os.path.join(RESUME_DIR, "*.txt")):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    resume_id = os.path.basename(path)
    resume_records.append({"resume_id": resume_id, "resume_text": text})

resumes_df_raw = pd.DataFrame(resume_records)

# Assuming jd_df_raw is already loaded successfully in a previous cell
jd_df_raw.head(), resumes_df_raw.head()

(                         job_title                         company  \
 0            Cybersecurity Analyst                      FIS Global   
 1  Associate Cybersecurity Analyst                          Unisys   
 2    Junior Cyber Security Analyst                          Endava   
 3          Junior Security Analyst  GrowthsphereAI Private Limited   
 4            Cybersecurity Analyst              Barracuda Networks   
 
                       location  \
 0  Bengaluru, Karnataka, India   
 1  Bengaluru, Karnataka, India   
 2  Bengaluru, Karnataka, India   
 3  Bengaluru, Karnataka, India   
 4  Bengaluru, Karnataka, India   
 
                                          description  
 0  FIS Global is hiring a Cybersecurity Analyst f...  
 1  Unisys is hiring Associate Cybersecurity Analy...  
 2  Endava is looking for a Junior Cyber Security ...  
 3  GrowthsphereAI Private Limited in Bengaluru is...  
 4  Barracuda Networks is seeking a Cybersecurity ...  ,
                       

In [38]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt') 
# Note: If you have a very recent version of NLTK and it still complains about the tokenizer, add: nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [39]:
import nltk
# Required for the tokenizer in newer NLTK versions
nltk.download('punkt_tab') 

# Often required for the lemmatizer in newer versions
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [40]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk, re

EN_STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_resume_sections(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text
    t = re.sub(r"\S+@\S+", " ", t)
    t = re.sub(r"\+?\d[\d\-\s]{7,}\d", " ", t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    lines = []
    for line in t.splitlines():
        ls = line.strip()
        if len(ls.split()) < 2:
            continue
        lines.append(ls)
    return " ".join(lines)

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = clean_resume_sections(text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = nltk.word_tokenize(text)
    cleaned = []
    for tok in tokens:
        if tok in EN_STOPWORDS:
            continue
        lemma = lemmatizer.lemmatize(tok)
        if lemma:
            cleaned.append(lemma)
    return " ".join(cleaned)

In [41]:
jd_df = jd_df_raw.copy()
resumes_df = resumes_df_raw.copy()

jd_df["clean_text"] = jd_df["description"].astype(str).apply(normalize_text)
resumes_df["clean_text"] = resumes_df["resume_text"].astype(str).apply(normalize_text)

jd_df[["job_title", "clean_text"]].head(), resumes_df[["resume_id", "clean_text"]].head()

(                         job_title  \
 0            Cybersecurity Analyst   
 1  Associate Cybersecurity Analyst   
 2    Junior Cyber Security Analyst   
 3          Junior Security Analyst   
 4            Cybersecurity Analyst   
 
                                           clean_text  
 0  fis global hiring cybersecurity analyst manage...  
 1  unisys hiring associate cybersecurity analyst ...  
 2  endava looking junior cyber security analyst j...  
 3  growthsphereai private limited bengaluru hirin...  
 4  barracuda network seeking cybersecurity analys...  ,
                                            resume_id  \
 0  0-yoe-unemployed-digital-marketing-india-revie...   
 1                                       12674256.txt   
 2                                       15603319.txt   
 3                                       16172429.txt   
 4                                       16507693.txt   
 
                                           clean_text  
 0  q ee contact n linkedin

In [ ]:
from collections import Counter
import re

skill_counter = Counter()

# From JD CSV 'skills' column if present
if "skills" in jd_df_raw.columns:
    for row in jd_df_raw["skills"].dropna():
        if isinstance(row, str):
            for s in re.split(r"[;,/]", row):
                s = s.strip()
                if s:
                    skill_counter[s.lower()] += 1

# Curated skills
curated_skills = [
    "python", "sql", "r", "go", "golang", "java", "c++", "c#", "typescript", "rust", 
    "kotlin", "swift", "dart", "php", "ruby", "perl", "bash", "powershell", "scala", 
    "javascript", "html", "css", "machine learning", "deep learning", "statistics", 
    "pandas", "numpy", "scikit-learn", "tensorflow", "pytorch", "keras", "xgboost", 
    "lightgbm", "catboost", "hugging face", "langchain", "langgraph", "transformers", 
    "bert", "llama 3", "openai gpt", "claude sonnet", "gemini flash", "rag", 
    "prompt engineering", "nlp", "computer vision", "opencv", "mediapipe", 
    "reinforcement learning", "supervised learning", "unsupervised learning", 
    "pydantic", "ollama", "fastapi", "django", "flask", "nestjs", "spring boot", 
    ".net 8+", "restful api design", "graphql", "microservices", "grpc", 
    "asynchronous logic", "node.js", "angular", "react", "vue.js", "tailwind css 4", 
    "bootstrap", "flutter", "jetpack compose", "kotlin multiplatform", "kmp", 
    "impeller engine", "provider", "riverpod", "bloc", "ktor", "firebase", 
    "responsive design", "shadcn/ui", "apache spark", "pyspark", "mllib", "hadoop", 
    "apache kafka", "amazon kinesis", "apache flink", "snowflake", "dbt", "airflow", 
    "jinja", "aws", "azure", "gcp", "docker", "kubernetes", "terraform", "pulumi", 
    "ansible", "jenkins", "github actions", "gitlab ci/cd", "argocd", "gitops", 
    "helm", "prometheus", "grafana", "elk stack", "opentelemetry", "datadog", 
    "cloudwatch", "infrastructure as code", "siem", "splunk", "microsoft sentinel", 
    "ibm qradar", "xdr", "edr", "crowdstrike", "carbon black", "ids", "ips", "nmap", 
    "nessus", "qualys", "rapid7", "openvas", "penetration testing", "forensic analysis", 
    "autopsy", "ftk imager", "encase", "misp", "mitre att&ck", "threat hunting", 
    "incident response", "malware analysis", "zero trust", "pci dss", "hipaa", 
    "gdpr", "nist csf", "figma", "adobe xd", "sketch", "axure rp", "balsamiq", 
    "invision", "photoshop", "illustrator", "after effects", "premiere pro", 
    "blender", "autodesk maya", "cinema 4d", "houdini", "unreal engine 5", "unity", 
    "substance 3d painter", "zbrush", "typography", "color theory", "wcag accessibility", 
    "design systems", "motion graphics", "vfx", "runway ml", "adobe sensei", "nuke", 
    "toon boom harmony", "ga4", "looker studio", "power bi", "tableau", "google ads", 
    "meta ads", "semrush", "ahrefs", "moz", "google search console", "seo", "sem", 
    "content marketing", "social media marketing", "email marketing", "hubspot", 
    "salesforce", "mailchimp", "marketo", "lead generation", "crm management", 
    "performance marketing", "roas analysis", "hotjar", "sprout social", "hootsuite", 
    "business analysis", "forecasting", "jira", "confluence", "trello", "asana", 
    "waterfall", "agile", "scrum", "kanban", "lean six sigma", "swot analysis", 
    "pestle analysis", "vrio framework", "porter's five forces", "ansoff matrix", 
    "bcg matrix", "cost-benefit analysis", "okr", "balanced scorecard", "hris", 
    "workday", "bamboohr", "adp", "ats", "talent sourcing", "onboarding", 
    "employer branding", "workforce metrics", "statutory compliance", "labor law", 
    "negotiation", "cultural intelligence", "financial modeling", "excel", 
    "power query", "pivot tables", "risk assessment", "aml", "kyc", 
    "transaction monitoring", "actimize", "sas", "oracle mantas", "fenergo", 
    "discounted cash flow", "quantitative analysis", "credit risk", "market risk", 
    "operational risk", "articulate storyline 360", "adobe captivate", "camtasia", 
    "vyond", "moodle", "canvas", "talentlms", "addie model", "bloom's taxonomy", 
    "xapi", "learning record stores", "microlearning", "qualitative analysis", 
    "literature review", "spss", "stata", "survey design", "data cleaning", 
    "hypothesis testing", "bayesian statistics", "ethnographic studies"
]
for s in curated_skills:
    skill_counter[s.lower()] += 3

min_freq = 3
skill_list = sorted([s for s, c in skill_counter.items() if c >= min_freq])
len(skill_list), skill_list[:20]

(270,
 ['.net 8+',
  'actimize',
  'addie model',
  'adobe captivate',
  'adobe sensei',
  'adobe xd',
  'adp',
  'after effects',
  'agile',
  'ahrefs',
  'airflow',
  'amazon kinesis',
  'aml',
  'angular',
  'ansible',
  'ansoff matrix',
  'apache flink',
  'apache kafka',
  'apache spark',
  'argocd'])

In [43]:
def extract_skills_from_text(text: str, skills) -> list:
    if not isinstance(text, str):
        return []
    t = text.lower()
    found = []
    for skill in skills:
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, t):
            found.append(skill)
    seen = set()
    unique = []
    for s in found:
        if s not in seen:
            seen.add(s)
            unique.append(s)
    return unique

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_corpus = pd.concat(
    [jd_df["clean_text"], resumes_df["clean_text"]],
    ignore_index=True
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2
)

tfidf_all = vectorizer.fit_transform(combined_corpus)

n_jd = len(jd_df)
jd_vectors = tfidf_all[:n_jd]
resume_vectors = tfidf_all[n_jd:]

In [45]:
category_mapping = {
    # --- Security Group ---
    'Cybersecurity Analyst': 'Security', 'Associate Cybersecurity Analyst': 'Security',
    'Junior Cyber Security Analyst': 'Security', 'Junior Security Analyst': 'Security',
    'Security Analyst': 'Security', 'Junior Cybersecurity Analyst': 'Security',
    'Entry Level Cybersecurity Analyst': 'Security', 'SOC Analyst': 'Security',
    'Cybersecurity Engineer (Entry Level)': 'Security',

    # --- Marketing & Sales Group ---
    'Digital Marketing Trainee': 'Marketing', 'E-Commerce Specialist': 'Marketing',
    'E-Commerce Intern': 'Marketing', 'Social Media Intern': 'Marketing',
    'SEO Intern': 'Marketing', 'Associate - Digital Marketing': 'Marketing',
    'Digital Media Specialist': 'Marketing', 'Digital Marketing Executive': 'Marketing',
    'Sales Executive - Digital Marketing & Studio Services': 'Marketing', 
    'SEO Executive': 'Marketing', 'Sales Executive - Digital Marketing Agency': 'Marketing',
    'Sales and Marketing Executive': 'Marketing', 'Digital Marketing - Fresher': 'Marketing',
    'Junior Digital Marketing Executive': 'Marketing',

    # --- HR & Recruitment Group ---
    'HR Executive (Generalist)': 'HR', 'HR & Admin Executive': 'HR',
    'Executive - HR Operations (Fresher)': 'HR', 'HR Executive': 'HR',
    'HR Documentation Executive': 'HR', 'Human Resource Intern': 'HR',
    'Human Resource Executive': 'HR', 'Human Resources (HR) Executive': 'HR',
    'HR Executive - Talent Acquisition': 'HR',

    # --- Data Science & Analytics Group ---
    'Data Scientist': 'Data Science/Analytics', 'Associate Data Scientist': 'Data Science/Analytics',
    'Graduate Data Scientist': 'Data Science/Analytics', 'Data Science Trainee': 'Data Science/Analytics',
    'Junior Data Scientist': 'Data Science/Analytics', 'Data Scientist Trainee': 'Data Science/Analytics',
    'Data Analyst': 'Data Science/Analytics', 'Business Intelligence Analyst': 'Data Science/Analytics',
    'Strategy Analyst': 'Data Science/Analytics',

    # --- Engineering & App Development Group ---
    'Systems Engineer': 'Engineering', 'DevOps Engineer - Mobility': 'Engineering',
    'Android Engineer': 'Engineering', 'Mobile App Development Intern': 'Engineering',
    'Android Developer': 'Engineering', 'Flutter Developer Intern': 'Engineering',
    'Flutter Development Intern': 'Engineering', 'Junior Flutter Developer': 'Engineering',
    'Flutter Developer': 'Engineering',

    # --- Design & UI/UX Group ---
    'Junior UI Designer': 'Design', 'UI Designer (Entry Level)': 'Design', 
    'Associate UI Designer': 'Design', 'Junior Visual / UI Designer': 'Design', 
    'UI Designer (Fresher)': 'Design',

    # --- Finance & Operations Group ---
    'Financial Sales Consultant': 'Finance', 'Budget Analyst': 'Finance',
    'Financial Institution Examiner': 'Finance', 'Operations Manager': 'Finance/Ops',
    'Associate Operations Manager': 'Finance/Ops',

    #Tech
    'Software Engineer': 'Engineering',
    'Full Stack Developer': 'Engineering',
    'Backend Developer': 'Engineering',
    'Frontend Developer': 'Engineering',
    'Java Developer': 'Engineering',
    'Python Developer': 'Engineering',
    'Machine Learning Engineer': 'Data Science/Analytics',
    'Data Engineer': 'Data Science/Analytics',
    'Cloud Engineer': 'Engineering',
    'Front End Web Developer': 'Engineering',
    'Full Stack Web Developer': 'Engineering',
    'B.Tech in Artificial Intelligence': 'Data Science/Analytics',
    'Research Assistant (AI/ML)': 'Data Science/Analytics'
}

# Mapping function with a cleaner default
def map_to_broad_category(title):
    return category_mapping.get(title, 'General Tech/Professional')

In [46]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# 1. Calculate similarity
sims = cosine_similarity(resume_vectors, jd_vectors)
best_idx = sims.argmax(axis=1)

# 2. Map the specific titles to broad categories
raw_pseudo_labels = jd_df["job_title"].iloc[best_idx].values
pseudo_labels_broad = pd.Series(raw_pseudo_labels).apply(map_to_broad_category)

X = resume_vectors
y = pseudo_labels_broad

# 3. Split the data
# Now that we have fewer classes, stratification is much more likely to work!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Unique categories after mapping: {len(y.unique())}")
print(y.value_counts())

Unique categories after mapping: 7
General Tech/Professional    78
Data Science/Analytics       17
Marketing                    11
Engineering                   8
HR                            5
Finance/Ops                   4
Security                      4
Name: count, dtype: int64


In [47]:
# This will show you the titles that the model is labeling as 'Other'
# so you can add them to your mapping dictionary.
unmapped_titles = jd_df["job_title"].unique()
print("Total Unique Titles in JD Dataset:", len(unmapped_titles))
print("\nAdd these to your category_mapping dictionary to balance the data:")
print(unmapped_titles)

Total Unique Titles in JD Dataset: 217

Add these to your category_mapping dictionary to balance the data:
['Cybersecurity Analyst' 'Associate Cybersecurity Analyst'
 'Junior Cyber Security Analyst' 'Junior Security Analyst'
 'Security Analyst' 'Junior Cybersecurity Analyst'
 'Entry Level Cybersecurity Analyst' 'SOC Analyst'
 'Cybersecurity Engineer (Entry Level)' 'Digital Marketing Trainee'
 'E-Commerce Specialist' 'E-Commerce Intern' 'Social Media Intern'
 'SEO Intern' 'Associate – Digital Marketing' 'Digital Media Specialist'
 'Digital Marketing Executive'
 'Sales Executive – Digital Marketing & Studio Services' 'SEO Executive'
 'Sales Executive – Digital Marketing Agency'
 'Sales and Marketing Executive' 'Digital Marketing - Fresher'
 'Digital Marketing – Fresher' 'Junior Digital Marketing Executive'
 'HR Executive (Generalist)' 'HR & Admin Executive'
 'Executive - HR Operations (Fresher)' 'HR Executive'
 'HR Documentation Executive' 'Human Resource Intern'
 'Human Resource Executi

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def eval_clf(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    return dict(name=name, model=model, accuracy=acc, precision=prec, recall=rec, f1=f1)

results = []

lr = LogisticRegression(max_iter=200, n_jobs=-1, random_state=42)
results.append(eval_clf("Logistic Regression", lr))

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
results.append(eval_clf("Random Forest", rf))

svm = LinearSVC(C=1.0, random_state=42)
results.append(eval_clf("Linear SVM", svm))

# Cosine baseline
sims_test = cosine_similarity(X_test, jd_vectors)
best_idx_test = sims_test.argmax(axis=1)
cosine_preds = jd_df["job_title"].iloc[best_idx_test].values

acc = accuracy_score(y_test, cosine_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, cosine_preds, average="weighted", zero_division=0
)
results.append(dict(name="TF-IDF + Cosine (baseline)", model=None,
                    accuracy=acc, precision=prec, recall=rec, f1=f1))

pd.DataFrame(results)[["name", "accuracy", "precision", "recall", "f1"]]


=== Logistic Regression ===
                           precision    recall  f1-score   support

   Data Science/Analytics       1.00      0.33      0.50         3
              Engineering       0.00      0.00      0.00         2
              Finance/Ops       0.00      0.00      0.00         1
General Tech/Professional       0.64      1.00      0.78        16
                       HR       0.00      0.00      0.00         1
                Marketing       0.00      0.00      0.00         2
                 Security       0.00      0.00      0.00         1

                 accuracy                           0.65        26
                macro avg       0.23      0.19      0.18        26
             weighted avg       0.51      0.65      0.54        26


=== Random Forest ===
                           precision    recall  f1-score   support

   Data Science/Analytics       1.00      0.33      0.50         3
              Engineering       0.00      0.00      0.00         2
      

c:\Users\anish\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
c:\Users\anish\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
c:\Users\anish\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
c:\Users\anish\AppData\Loca

,name,accuracy,precision,recall,f1
0,Logistic Regression,0.653846,0.509231,0.653846,0.537992
1,Random Forest,0.692308,0.564103,0.692308,0.588462
2,Linear SVM,0.692308,0.602564,0.692308,0.601282
3,TF-IDF + Cosine (baseline),0.000000,0.000000,0.000000,0.000000


In [49]:
# import pickle

# # 1. Save the trained Random Forest model
# with open('best_rf_model.pkl', 'wb') as f:
#     pickle.dump(rf, f)

# # 2. Save the TF-IDF vectorizer 
# # (This is crucial to ensure the text features match!)
# with open('tfidf_vectorizer.pkl', 'wb') as f:
#     pickle.dump(vectorizer, f)

# # 3. Save the category mapping for later reference
# with open('category_mapping.pkl', 'wb') as f:
#     pickle.dump(category_mapping, f)

# print("Model and Vectorizer saved successfully!")

In [50]:
import pickle

# 1. Save the trained Linear SVM model
# We use the 'svm' variable which holds your LinearSVC instance
with open('best_svm_model.pkl', 'wb') as f:
    pickle.dump(svm, f)

# 2. Save the TF-IDF vectorizer 
# This must be the vectorizer that was fit on your training data
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# 3. Save the category mapping 
with open('category_mapping.pkl', 'wb') as f:
    pickle.dump(category_mapping, f)

print("SVM Model and Vectorizer saved successfully!")

SVM Model and Vectorizer saved successfully!
